# Notas — Aula 12: Metaprogramação

Marco: leituras de sensor em qualquer direção (`robo.leitura_norte`, `robo.leitura_sul`, ...)
passam a ser calculadas **sob demanda** via `__getattr__`, com cache guardado direto no
`__dict__` da instância; toda mudança de atributo passa a ficar **auditada** via `__setattr__`;
e toda subclasse de `Robo` passa a se registrar sozinha em `Robo._registro` via
`__init_subclass__` — sem lista mantida à mão.

In [176]:
from enum import Enum

LADO_GRADE = 10

## Módulos são objetos

Quando vocês escrevem `import math`, o que `math` passa a significar, de fato? Um
**objeto** — de tipo `module` — com atributos e métodos acessíveis por notação de ponto,
exatamente como `robo1.x`. `import` não é sintaxe mágica: é só o jeito da linguagem criar
(ou recuperar) esse objeto e ligar um nome a ele no escopo atual. Todo módulo carrega um
`__dict__` próprio, igual a qualquer outro objeto do curso.

In [177]:
import math

print(type(math))
print(math.pi)
print(sorted(math.__dict__.keys())[:5])
print(math.__name__)

<class 'module'>
3.141592653589793
['__doc__', '__loader__', '__name__', '__package__', '__spec__']
math


## `sys.modules`: o cache do import

Python só executa o corpo de um módulo **uma vez** por sessão do interpretador — da
segunda vez em diante, `import` devolve o mesmo objeto já criado, guardado num
dicionário global chamado `sys.modules` (nome do módulo → objeto módulo).

In [178]:
import sys

print("math" in sys.modules)
print(sys.modules["math"] is math)

True
True


## `if __name__ == "__main__"` — a dívida de todo starter

Todo `.py` que vocês rodam desde a D1 termina com `if __name__ == "__main__":`. Agora dá
pra explicar: quando um arquivo é executado diretamente, o Python atribui ao `__name__`
**daquele módulo** o valor especial `"__main__"`. Quando o mesmo arquivo é **importado**
por outro, `__name__` vira o nome do arquivo. O `if` pergunta, literalmente, 'estou sendo
o programa principal, ou fui só importado?' — e só roda o bloco no primeiro caso.

In [179]:
print(__name__)
print(math.__name__)

__main__
math


## `__getattr__`: um atributo que só existe quando alguém pergunta

Por padrão, pedir um atributo que não existe estoura `AttributeError`. Mas um objeto pode
implementar `__getattr__(self, nome_attr)` — um método chamado **só** quando a busca normal
(primeiro o `__dict__` da instância, depois a classe) falha. É a peça que faltava para
calcular algo sob demanda, em vez de sempre na criação do objeto.

In [180]:
class Pessoa:
    def __init__(self, nome):
        self.nome = nome


p = Pessoa("Ana")
print(p.nome)
try:
    p.idade
except AttributeError as erro:
    print(f"{type(erro).__name__}: {erro}")

Ana
AttributeError: 'Pessoa' object has no attribute 'idade'


In [181]:
class PessoaComGetattr:
    def __init__(self, nome):
        self.nome = nome

    def __getattr__(self, nome_attr):
        return f"{nome_attr} desconhecido"


p2 = PessoaComGetattr("Ana")
print(p2.nome)      # __getattr__ NÃO é chamado — já está no __dict__
print(p2.idade)      # __getattr__ é chamado — não existe em lugar nenhum

Ana
idade desconhecido


## Robô: leitura de sensor sob demanda

Em vez de escrever `leitura_norte()`, `leitura_sul()`, `leitura_leste()`,
`leitura_oeste()` — quatro métodos quase idênticos —, o robô deixa o **nome do atributo**
carregar a direção. `__getattr__` intercepta qualquer `robo.leitura_<direção>`, calcula se
a próxima casa está livre, e guarda o resultado num cache dentro de `self.__dict__` (nunca
`self.atributo = ...` — isso disparia `__getattr__` de novo, numa recursão infinita).

In [182]:
class Direcao(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)


class Robo:
    def __init__(self, x=0, y=0, obstaculos=None):
        self.x = x
        self.y = y
        self.obstaculos = obstaculos if obstaculos is not None else {}

    def __getattr__(self, nome_attr):
        cache = self.__dict__.setdefault("_cache_leituras", {})
        if nome_attr in cache:
            return cache[nome_attr]
        if nome_attr.startswith("leitura_"):
            direcao_nome = nome_attr.removeprefix("leitura_").upper()
            try:
                direcao = Direcao[direcao_nome]
            except KeyError:
                raise AttributeError(f"Robo não tem atributo {nome_attr!r}") from None #Returns with the error raising.
            dx, dy = direcao.value
            nx, ny = self.x + dx, self.y + dy
            livre = (0 <= nx < LADO_GRADE and 0 <= ny < LADO_GRADE
                     and (nx, ny) not in self.obstaculos)
            cache[nome_attr] = livre
            return livre
        raise AttributeError(f"Robo não tem atributo {nome_attr!r}")


robo1 = Robo(x=5, y=5)
print(robo1.leitura_norte)
print(robo1.leitura_sul)

True
True


### Sua vez

Complete `__getattr__` de `RoboComAqui` para tratar `"leitura_aqui"` como caso especial: a
posição atual, por definição, está sempre livre — devolva `True` direto, sem consultar
`Direcao`. Guarde no cache do mesmo jeito que os outros.

*Dica: um `if nome_attr == "leitura_aqui": cache[nome_attr] = True; return True` antes do*
*`if nome_attr.startswith("leitura_"):` resolve.*

In [183]:
class RoboComAqui:
    def __init__(self, x=0, y=0, obstaculos=None):
        self.x = x
        self.y = y
        self.obstaculos = obstaculos if obstaculos is not None else {}

    def __getattr__(self, nome_attr):
        cache = self.__dict__.setdefault("_cache_leituras", {})
        if nome_attr in cache:
            return cache[nome_attr]
        # TODO: se nome_attr == "leitura_aqui", guarde True no cache e devolva True
        if nome_attr == "leitura_aqui":
            cache[nome_attr] = True
            return True
        ...
        if nome_attr.startswith("leitura_"):
            direcao_nome = nome_attr.removeprefix("leitura_").upper()
            try:
                direcao = Direcao[direcao_nome]
            except KeyError:
                raise AttributeError(f"Robo não tem atributo {nome_attr!r}") from None
            dx, dy = direcao.value
            nx, ny = self.x + dx, self.y + dy
            livre = (0 <= nx < LADO_GRADE and 0 <= ny < LADO_GRADE
                     and (nx, ny) not in self.obstaculos)
            cache[nome_attr] = livre
            return livre
        raise AttributeError(f"Robo não tem atributo {nome_attr!r}")


robo2 = RoboComAqui(x=5, y=5)
try:
    print(robo2.leitura_aqui)
except AttributeError:
    print("Complete o TODO acima para ver o resultado.")

True


## `__setattr__`: auditando toda mudança de estado

`__getattr__` reage quando um atributo **não existe**; `__setattr__` é o oposto — roda para
**toda** atribuição, sempre, mesmo para atributos que já existem. Dá pra usar isso para
manter um histórico de mudanças, útil quando um teste falha e é preciso descobrir quem
mudou o quê.

In [184]:
class RoboAuditado:
    def __init__(self, x=0):
        self.x = x

    def __setattr__(self, nome_attr, valor):
        log = self.__dict__.setdefault("_log_mudancas", [])
        if nome_attr != "_log_mudancas":
            log.append((nome_attr, valor))
        super().__setattr__(nome_attr, valor)


robo3 = RoboAuditado()
robo3.x = 1
robo3.x = 2
print(robo3._log_mudancas)

[('x', 0), ('x', 1), ('x', 2)]


### Sua vez

O log de cima grava **tudo**, até atributos internos que começam com `"_"`. Complete
`__setattr__` de `RoboAuditadoFiltrado` para ignorar (não gravar no log) qualquer
`nome_attr` que comece com `"_"` — continue chamando `super().__setattr__(...)` para todo
mundo; só o **log** fica seletivo.

*Dica: `if not nome_attr.startswith("_"): log.append((nome_attr, valor))`.*

In [ ]:
class RoboAuditadoFiltrado:
    def __init__(self, x=0):
        self.x = x
        self._interno = 0

    def __setattr__(self, nome_attr, valor):
        log = self.__dict__.setdefault("_log_mudancas", [])
        # TODO: só chame log.append((nome_attr, valor)) se nome_attr NÃO começar com "_"
        ...
        if not nome_attr.startswith("_"): #Evita escrever atributos "ocultos"
            log.append((nome_attr, valor))
        super().__setattr__(nome_attr, valor)


robo4 = RoboAuditadoFiltrado()
print(robo4._log_mudancas)

[('x', 0)]


## `__init_subclass__`: toda subclasse se registra sozinha

Existe um hook chamado automaticamente toda vez que uma classe herda de outra:
`__init_subclass__(cls, **kwargs)`, definido na classe-mãe. Ele roda no momento da
**definição** da subclasse — antes de qualquer instância existir — e não precisa de
`@classmethod`: o Python já trata como método de classe. É assim que frameworks como
Django/pytest descobrem sozinhos as classes que vocês escrevem, sem registro manual.

In [186]:
class Animal:
    registrados = []

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Animal.registrados.append(cls)


class Cachorro(Animal):
    pass


class Gato(Animal):
    pass


print(Animal.registrados)

[<class '__main__.Cachorro'>, <class '__main__.Gato'>]


## O robô ganha um catálogo automático

O mesmo mecanismo aplicado ao `Robo`: qualquer subclasse — presente ou futura — se cadastra
sozinha em `Robo._registro`, sem ninguém manter uma lista à mão. `**kwargs` na própria
definição da classe (`class RoboVeloz(Robo, categoria="ofensivo"):`) deixa cada subclasse
anotar sua própria categoria.

In [187]:
class Robo3:
    _registro = {}

    def __init_subclass__(cls, categoria="geral", **kwargs):
        super().__init_subclass__(**kwargs)
        Robo3._registro[cls.__name__] = cls
        cls.categoria = categoria

    def __init__(self, nome):
        self.nome = nome


class RoboVeloz(Robo3, categoria="ofensivo"):
    pass


print(Robo3._registro)
print(RoboVeloz.categoria)

{'RoboVeloz': <class '__main__.RoboVeloz'>}
ofensivo


### Sua vez

Complete `__init_subclass__` de `Robo4` (mesma ideia do `Robo3` acima): falta guardar a
`categoria` recebida (ou o valor padrão) como atributo da subclasse.

*Dica: uma linha, `cls.categoria = categoria` — igual ao `Robo3`.*

In [188]:
class Robo4:
    _registro = {}

    def __init_subclass__(cls, categoria="geral", **kwargs):
        super().__init_subclass__(**kwargs)
        Robo4._registro[cls.__name__] = cls
        # TODO: atribua cls.categoria = categoria
        ...

    def __init__(self, nome):
        self.nome = nome


class RoboExplorador(Robo4):
    pass


print(Robo4._registro)
try:
    print(RoboExplorador.categoria)
except AttributeError:
    print("Complete o TODO acima para ver o resultado.")

{'RoboExplorador': <class '__main__.RoboExplorador'>}
Complete o TODO acima para ver o resultado.


## Para aprofundar

- Módulos (tutorial completo) — documentação oficial: https://docs.python.org/3/tutorial/modules.html
- `sys.modules` — documentação oficial: https://docs.python.org/3/library/sys.html#sys.modules
- `if __name__ == "__main__"` — documentação oficial: https://docs.python.org/3/library/__main__.html
- Customização de acesso a atributo (`__getattr__`, `__setattr__`) — referência oficial do modelo de dados: https://docs.python.org/3/reference/datamodel.html#customizing-attribute-access
- `__getattr__` em nível de módulo (PEP 562) — documento da PEP: https://peps.python.org/pep-0562/
- `__init_subclass__` — referência oficial do modelo de dados: https://docs.python.org/3/reference/datamodel.html#object.__init_subclass__
- Metaclasses (leitura avançada) — Real Python: https://realpython.com/python-metaclasses/